# Home task: data engineering

## Phase 0: Preparation and Financial Safety

### 1. AWS Free Tier Account Creation

An AWS Free Tier account was successfully created using an email address and a virtual card with a minimal balance for verification purposes.

To improve account security and avoid unexpected charges:
- Multi-Factor Authentication (MFA) was enabled.
- AWS Budget alerts were configured to monitor spending.
- Free Tier usage limits were reviewed prior to using any AWS services.

### 2. Billing Alarm Configuration

A CloudWatch Billing Alarm named `billing-alerts-roman` was created to monitor AWS charges.  
The alarm was configured with the following condition:

- `EstimatedCharges > 1 USD`

Amazon SNS email notifications were configured to deliver billing alerts to the registered email address.  
The SNS subscription was confirmed successfully via the email confirmation link.

![CloudWatch billing alarm](images/Screenshot1ActiveCloudWatchalarmfor1.png)

## Phase 1: Identity and Access Management (IAM)

A new IAM user was created to work with AWS services instead of using the root account.  
The user was named `data_engineer_roman`.

The following security and access settings were applied:
- AWS Management Console access was enabled.
- The `AdministratorAccess` policy was attached to the user.
- Multi-Factor Authentication (MFA) was enabled (Virtual MFA device).
- The initial sign-in was completed successfully using the IAM user credentials.


![IAM users list](images/Screenshot3IAMusers.png)

## Phase 2: Building a Data Lake with S3, Glue and Athena

### 2.1 S3: Data Storage

An S3 bucket named `student-data-lake-roman-2026` was created in the **Europe (Stockholm)** region to store source data for the Data Lake pipeline.

A CSV file named `students_data.csv` was uploaded to the bucket successfully (199.0 B, 100% success rate).

![Uploaded CSV file in S3 bucket](images/Screenshot2uploadedfileinS3Bucket.png)

### 2.2 IAM Role: Access for AWS Glue

An IAM role was created for AWS Glue to grant access to S3 and the Glue Data Catalog.

This role allows AWS Glue to read the uploaded CSV file from the S3 bucket and register metadata in the Glue Data Catalog.  
The following policies were attached to the role:

- `AmazonS3FullAccess`
- `AWSGlueServiceRole`

This role was subsequently used by the Glue Crawler to scan the S3 data and automatically infer a table schema.

### 2.3 AWS Glue Crawler and Data Catalog

An AWS Glue Crawler was created and configured to scan the CSV file stored in the S3 bucket `student-data-lake-roman-2026`.

The crawler used the target database `students_data_db`. After running, AWS Glue automatically detected the schema and created a table in the Glue Data Catalog.

The resulting table was named `student_data_lake_roman_2026`. The schema includes the following columns:
- `student_id` — `bigint`
- `name` — `string`
- `course` — `string`
- `grade` — `bigint`
- `city` — `string`


![Glue Data Catalog table schema](images/5.png)

### 2.4 Amazon Athena: Querying Data

Amazon Athena was used to query the table created by the AWS Glue Crawler.

The database `students_data_db` and the table `student_data_lake_roman_2026` were selected in the Athena Query Editor.  
The following SQL query was executed:

```sql
SELECT *
FROM students_data_db.student_data_lake_roman_2026
LIMIT 10;
```

The query completed successfully (Run time: 389 ms, Data scanned: 0.19 KB) and returned 5 rows with student records including fields: `student_id`, `name`, `course`, `grade`, and `city`.

![Athena query results](images/SC4.jpg)

## Phase 3: LocalStack and AWS CLI (Optional)

As an optional part of the task, LocalStack was used to simulate AWS services locally through Docker and AWS CLI.

LocalStack was started in a Docker container using the image `localstack/localstack:4.4.0`.  
After the service became ready, AWS CLI commands were executed against the LocalStack endpoint:

```bash
aws --endpoint-url=http://localhost:4566 s3 mb s3://test-bucket
aws --endpoint-url=http://localhost:4566 s3 cp data.csv s3://test-bucket
aws --endpoint-url=http://localhost:4566 s3 ls s3://test-bucket
```

![LocalStack S3 commands](images/pw.png)